# Customer Churn Prediction

This notebook develops a reproducible classification pipeline for identifying
telecom customers at risk of leaving. It uses IBM's fictional Telco Customer
Churn sample with 7,043 customer records.

**Business objective:** prioritize customers for retention outreach while
detecting as many true churners as reasonably possible.


## Workflow

1. Load and validate the dataset
2. Inspect target balance and data quality
3. Split training and test data
4. Build a leakage-safe preprocessing pipeline
5. Train class-balanced logistic regression
6. Evaluate holdout and cross-validated performance
7. Interpret the strongest model signals


In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.churn_pipeline import RANDOM_STATE, build_pipeline, coefficient_table, load_data


## 1. Load and validate the data


In [2]:
data_path = PROJECT_ROOT / "data" / "raw" / "Telco-Customer-Churn.csv"
X, y, cleaned = load_data(data_path)

print(f"Rows: {len(cleaned):,}")
print(f"Input features: {X.shape[1]}")
print(f"Churned customers: {y.sum():,} ({y.mean():.1%})")
print(f"Missing TotalCharges values: {cleaned['TotalCharges'].isna().sum()}")


Rows: 7,043
Input features: 19
Churned customers: 1,869 (26.5%)
Missing TotalCharges values: 11


In [3]:
cleaned[[
    "tenure", "Contract", "InternetService",
    "MonthlyCharges", "TotalCharges", "Churn"
]].head()


## 2. Understand the target

Only 26.5% of customers churned, so accuracy alone can be misleading. A model
that predicts "stayed" for everyone would already be about 73.5% accurate.

![Customer churn distribution](../reports/figures/churn_distribution.png)


## 3. Split the data before preprocessing


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")


Training rows: 5,634
Test rows: 1,409


## 4. Build and train the pipeline

The pipeline learns imputation, scaling, and encoding values using only the
training data. Numeric features use median imputation and standardization.
Categorical features use most-frequent imputation and one-hot encoding.
Unknown categories are ignored safely during prediction.


In [5]:
model = build_pipeline()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]


## 5. Evaluate holdout performance


In [6]:
holdout_metrics = pd.Series({
    "Accuracy": accuracy_score(y_test, predictions),
    "Precision": precision_score(y_test, predictions),
    "Recall": recall_score(y_test, predictions),
    "F1": f1_score(y_test, predictions),
    "ROC-AUC": roc_auc_score(y_test, probabilities),
})

print(holdout_metrics.round(3).to_string())


Accuracy     0.738
Precision    0.504
Recall       0.783
F1           0.614
ROC-AUC      0.841


The model identified **293 of 374 actual churners** in the test set. Its high
recall is useful when missing a churner is more costly than contacting a
customer who would have stayed.

![Confusion matrix](../reports/figures/confusion_matrix.png)

![ROC curve](../reports/figures/roc_curve.png)


## 6. Check stability with cross-validation


In [7]:
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

cv_auc = cross_val_score(
    build_pipeline(),
    X_train,
    y_train,
    scoring="roc_auc",
    cv=cross_validation,
)

print("Fold ROC-AUC:", cv_auc.round(3))
print(f"Mean ROC-AUC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")


Fold ROC-AUC: [0.847 0.825 0.842 0.863 0.853]
Mean ROC-AUC: 0.846 ± 0.012


## 7. Interpret model signals


In [8]:
top_signals = coefficient_table(model).head(10)
print(top_signals[["feature", "coefficient"]].round(3).to_string(index=False))


                             feature  coefficient
                              tenure       -1.140
                   Contract_Two year       -0.783
         InternetService_Fiber optic        0.702
                      MonthlyCharges       -0.673
             Contract_Month-to-month        0.653
                 InternetService_DSL       -0.629
                        TotalCharges        0.473
     TechSupport_No internet service       -0.281
DeviceProtection_No internet service       -0.281
                  InternetService_No       -0.281


![Strongest logistic-regression signals](../reports/figures/top_coefficients.png)

Longer tenure and two-year contracts are associated with lower predicted churn,
while month-to-month contracts and fiber-optic service are associated with
higher predicted churn. These are model associations, not proof of causation.


## Conclusion

The pipeline achieves a holdout ROC-AUC of 0.841 and detects 78.3% of true
churners. Cross-validation results are consistent across folds, suggesting the
result is not dependent on one lucky split.

### Limitations

- The dataset describes a fictional telecom company.
- A real deployment needs monitoring, probability calibration, and drift checks.
- The classification threshold should be chosen using actual retention costs.
- Coefficients describe associations and should not be treated as causal effects.
